# Visual comparison of polarisation synthesis in SAR and geographic coordinates

- This is only to check the output for geocoded images.
- Numerical comparison is done in the real-data parity notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
from dask.diagnostics import ProgressBar

from polsarpro.io import open_netcdf_beam
from polsarpro.polarisation import polarisation_synthesis
from polsarpro.util import S_to_T3, boxcar, multilook

input_alos_slc = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")
input_alos_geo = Path(
    "/data/psp/test_files/SAN_FRANCISCO_ALOS1_geocoded_T3_7_look_az.nc"
)

In [ ]:
slc = open_netcdf_beam(input_alos_slc)
geo = open_netcdf_beam(input_alos_geo)

In [ ]:
with ProgressBar():
    T3_slc = S_to_T3(slc)
    T3_slc = multilook(T3_slc, dim_az=4, dim_rg=1)
    T3_slc = boxcar(T3_slc, dim_az=3, dim_rg=3)
    res_slc = polarisation_synthesis(
        T3_slc, phi=17.0, tau=-11.0, basis="sinclair"
    ).persist()
with ProgressBar():
    res_geo = polarisation_synthesis(
        geo, phi=17.0, tau=-11.0, basis="sinclair"
    ).persist()

In [ ]:
high_slc = res_slc.quantile(0.98, dim=("y", "x")).drop_vars("quantile")
high_geo = res_geo.quantile(0.98, dim=("lat", "lon")).drop_vars("quantile")
rgb_slc = (res_slc / high_slc).clip(0, 1)
rgb_geo = (res_geo / high_geo).clip(0, 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 8))
axes[0].imshow(
    rgb_slc.transpose("y", "x", "band")[::2], interpolation="none"
)
axes[0].set_title("SAR coordinates")
axes[0].axis("off")
axes[1].imshow(
    rgb_geo.transpose("lat", "lon", "band"), interpolation="none"
)
axes[1].set_title("Geographic coordinates")
axes[1].axis("off")
plt.tight_layout()